In [1]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

try:
    import irksome
except ImportError:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git#egg=Irksome
    import irksome

Block triangular preconditioners for the heat equation
====================================================

This demo applies the method suggested in:

Mardal, Nilssen, Staff, "Order-optimal preconditioners for implicit
Runge-Kutta schemes applied to parabolic PDEs", SISC 29(1): 361--375 (2007),

to our ongoing heat equation demonstration problem on $\Omega = [0,10]
\times [0,10]$, with boundary $\Gamma$, giving rise to the weak form

$$
(u_t, v) + (\nabla u, \nabla v) = (f, v)
$$

A multi-stage RK method applied to the heat equation gives a
block-structured system.  The on-diagonal blocks are quite similar to
what one obtains from a backward Euler discretization of the equation.

With a 2-stage method, we have

$$
\left[ \begin{array}{cc}
M + a_{11} \Delta t K &  a_{12} \Delta t K \\
a_{21} \Delta t K & M + a_{22} \Delta t K
\end{array}
\right]
\left[ \begin{array}{c} k_1 \\ k_2 \end{array} \right]
   =
\left[ \begin{array}{cc} A_{11} & A_{12} \\ A_{21} & A_{22} \end{array} \right]
   \left[ \begin{array}{c} k_1 \\ k_2 \end{array} \right]
   = \left[ \begin{array}{c} f_1 \\ f_2 \end{array} \right]
$$



And the suggestion (analyzed rigorously) of Mardal, Nilssen, and Staff
is to use a block diagonal preconditioner:

$$
P = \left[ \begin{array}{cc} A_{11} & 0 \\ 0 & A_{22} \end{array} \right]
$$


This allows one to leverage an existing methodology for a low order
method like backward Euler for the diagonal blocks.  In our case, we
will simply use an algebraic multigrid scheme, although one could
certainly use geometric multigrid or some other technique.

In [2]:
from firedrake import *  # noqa: F403
from irksome import LobattoIIIC, TimeStepper, Dt

Here is our setup as in the previous demo:

In [3]:
butcher_tableau = LobattoIIIC(2)
N = 64
x0 = 0.0
x1 = 10.0
y0 = 0.0
y1 = 10.0
msh = RectangleMesh(N, N, x1, y1)
dt = Constant(10. / N)
t = Constant(0.0)
V = FunctionSpace(msh, "CG", 1)
x, y = SpatialCoordinate(msh)
S = Constant(2.0)
C = Constant(1000.0)
B = (x-Constant(x0))*(x-Constant(x1))*(y-Constant(y0))*(y-Constant(y1))/C
R = (x * x + y * y) ** 0.5
uexact = B * atan(t)*(pi / 2.0 - atan(S * (R - t)))
rhs = Dt(uexact) - div(grad(uexact))
u = Function(V)
u.interpolate(uexact)
v = TestFunction(V)
F = inner(Dt(u), v)*dx + inner(grad(u), grad(v))*dx - inner(rhs, v)*dx
bc = DirichletBC(V, 0, "on_boundary")

Now, we define the solver parameters.  PETSc-speak for taking the
block diagonal is an "additive fieldsplit", and we specify just using PETSc's algebraic multigrid on all of the blocks

In [4]:
params = {
    "mat_type": "aij",
    "snes_type": "ksponly",
    "ksp_type": "gmres",
    "ksp_monitor": None,
    "pc_type": "fieldsplit",   # block preconditioner
    "pc_fieldsplit_type": "additive",  # block diagaonal
    "fieldsplit": {
    "ksp_type": "preonly",
        "pc_type": "gamg"
    }
}

Note that we have used the same technique for each RK stage, which is
probably typical.  However, it is not necessary at all.
     
To test this preconditioning strategy, we'll create a time stepping
object which will set up the variational problem for us::

In [5]:
stepper = TimeStepper(F, butcher_tableau, t, dt, u, bcs=bc,
                      solver_parameters=params)

Since we're just testing the efficacy of the preconditioner, we'll just take one step:

In [6]:
stepper.advance()

    Residual norms for firedrake_1_ solve.
    0 KSP Residual norm 2.370167625025e+00
    1 KSP Residual norm 7.521537865473e-02
    2 KSP Residual norm 2.449435553330e-02
    3 KSP Residual norm 1.404113546815e-02
    4 KSP Residual norm 5.165488219474e-03
    5 KSP Residual norm 2.631609621111e-03
    6 KSP Residual norm 1.204298130967e-03
    7 KSP Residual norm 5.764952188071e-04
    8 KSP Residual norm 2.708319036705e-04
    9 KSP Residual norm 1.309874413193e-04
   10 KSP Residual norm 6.399328352248e-05
   11 KSP Residual norm 3.050203812618e-05
   12 KSP Residual norm 1.524883369102e-05


However, this preconditioner does not scale with the number of stages:

In [7]:
stepper = TimeStepper(F, LobattoIIIC(5), t, dt, u, bcs=bc,
                      solver_parameters=params)
stepper.advance()

    Residual norms for firedrake_2_ solve.
    0 KSP Residual norm 3.682822778039e+00
    1 KSP Residual norm 1.258536926114e-01
    2 KSP Residual norm 5.467110580990e-02
    3 KSP Residual norm 3.501011054821e-02
    4 KSP Residual norm 1.895717384044e-02
    5 KSP Residual norm 1.354380829402e-02
    6 KSP Residual norm 7.491541601035e-03
    7 KSP Residual norm 5.218365529304e-03
    8 KSP Residual norm 4.338461466538e-03
    9 KSP Residual norm 3.073297109535e-03
   10 KSP Residual norm 2.247056889038e-03
   11 KSP Residual norm 1.737307259243e-03
   12 KSP Residual norm 1.466286875637e-03
   13 KSP Residual norm 1.340498578382e-03
   14 KSP Residual norm 9.738625261556e-04
   15 KSP Residual norm 6.788779314602e-04
   16 KSP Residual norm 5.931627862533e-04
   17 KSP Residual norm 4.532219887513e-04
   18 KSP Residual norm 3.576579707615e-04
   19 KSP Residual norm 2.913970532173e-04
   20 KSP Residual norm 2.185315770594e-04
   21 KSP Residual norm 1.945072994067e-04
   22 KSP R

Rana *et al* (SISC 2021) propose a different approach to block triangular preconditioning that overcomes this difficulty.  It rediscretizes the time-dependent problem with a block-triangular approximation to the Butcher matrix, which leads to a block triangular system.  One approach is to take the standard factorization $A = LDU$, and then define
$$
\tilde{A} = LD.
$$

This is implemented quite generally in Irksome as Python preconditioner (built on top of the auxiliary operator framework).  Here, we make the Rana approximation, then do a multiplicative fieldsplit and approximate the inverses of the diagonal blocks with a sweep of AMG:

In [8]:
ranaparams = {
    "mat_type": "aij",
    "snes_type": "ksponly",
    "ksp_type": "gmres",
    "ksp_monitor": None,
    "pc_type": "python",
    "pc_python_type": "irksome.RanaLD",
    "aux": {
        "pc_type": "fieldsplit",
        "pc_fieldsplit_type": "multiplicative",
        "fieldsplit": {
            "ksp_type": "preonly",
            "pc_type": "gamg"
        }
    }
}

stepper = TimeStepper(F, LobattoIIIC(5), t, dt, u, bcs=bc,
                      solver_parameters=ranaparams)
stepper.advance()


    Residual norms for firedrake_3_ solve.
    0 KSP Residual norm 3.432274496723e+00
    1 KSP Residual norm 7.457486743103e-02
    2 KSP Residual norm 1.762660528694e-02
    3 KSP Residual norm 6.255425845097e-03
    4 KSP Residual norm 1.684734631772e-03
    5 KSP Residual norm 8.967218614118e-04
    6 KSP Residual norm 5.619831482621e-04
    7 KSP Residual norm 3.465515760737e-04
    8 KSP Residual norm 1.748856199976e-04
    9 KSP Residual norm 7.832523219586e-05
   10 KSP Residual norm 3.483446306498e-05
   11 KSP Residual norm 1.851650044342e-05


It is important to note that the cost of applying the Rana preconditioner per iteration is essentially the same as the original multiplicative field split, but we require many fewer outer iterations.
